In [ ]:
# conda activate dream

library(edgeR)
library(dplyr)
library(parallelly)
library(data.table)
library(BiocParallel)
library(variancePartition)

setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/pseudobulk_test/yao_2021/MOp")

Sys.setenv(OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
if (requireNamespace("RhpcBLASctl", quietly=TRUE)) {
  RhpcBLASctl::blas_set_num_threads(1)
  RhpcBLASctl::omp_set_num_threads(1)
}

param <- SnowParam(workers=6, type="SOCK", progressbar=TRUE)

Here I run DE analysis using dream on the pseudobulked cell type data

In [ ]:
counts <- fread("data/yao_2021_MOp_STAR_donor_cell_type_pseudobulk.csv", data.table=FALSE)
sample_meta <- fread("data/yao_2021_MOp_STAR_donor_cell_type_pseudobulk_sampleinfo.csv", data.table=FALSE)

In [ ]:
# Add sex and age info. to metadata
sample_meta$Row <- 1:nrow(sample_meta)
donor_meta <- fread("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/scRNA-seq/yao_2021/yao_2021_tableS2_sampleinfo_donor_level.csv", data.table=FALSE)
donor_meta <- donor_meta[donor_meta$region_label %in% "MOp",]
sample_meta <- merge(sample_meta, donor_meta, by.x="Donor", by.y="external_donor_name", all.x=TRUE, sort=FALSE)
sample_meta <- sample_meta[order(sample_meta$Row),] # Keep original row order
rownames(sample_meta) <- sample_meta$Sample_ID

# Reformat cell type names
sample_meta$Cell_type <- sapply(sample_meta$Cell_type, function(x) gsub(" ", "_", x))
sample_meta$Cell_type <- sapply(sample_meta$Cell_type, function(x) gsub("/", "_", x, fixed=TRUE))

sample_meta$Sex <- sample_meta$sex
sample_meta$Age <- as.numeric(gsub("P", "", sample_meta$age))

# Center age
sample_meta$Age_c <- as.numeric(scale(sample_meta$Age, center=TRUE, scale=FALSE))

# Prep count data

y <- DGEList(counts) 
keep <- filterByExpr(y, group=sample_meta$Cell_type)
y <- y[keep,, keep.lib.sizes=FALSE]
print(dim(y$counts))
# > print(dim(y$counts))
# [1] 24133   229
y <- calcNormFactors(y)

ctypes <- unique(sample_meta$Cell_type)
ctype_levels <- levels(factor(ctypes))

In [ ]:
# # Subset data for testing

# set.seed(1)
# sample_idx <- sample(1:nrow(sample_meta), size=200)
# sample_meta <- sample_meta[sample_idx,]
# counts_subset <- counts[sample(1:nrow(counts), size=1000), c(1, sample_idx + 1)]

# y <- DGEList(counts_subset)
# keep <- filterByExpr(y, group=sample_meta$Cell_type)
# y <- y[keep,, keep.lib.sizes=FALSE]
# dim(y$counts)
# y <- calcNormFactors(y)

# ctypes <- unique(sample_meta$Cell_type)
# ctype_levels <- levels(factor(ctypes))

## Pairwise tests

Compare model coefficients expression between each target cell type each other cell type (pairwise)

In [ ]:
form <- ~ 0 + Cell_type + Sex + Age_c + (1 | Donor)
vobj <- voomWithDreamWeights(y, form, sample_meta, BPPARAM=param)

In [ ]:
form_fixed <- ~ 0 + Cell_type + Sex + Age_c
coef_names <- colnames(model.matrix(form_fixed, data=sample_meta))
ctype_coef_names <- coef_names[-c(length(coef_names)-1, length(coef_names))]
ctype_pairs <- combn(ctype_coef_names, m=2) 

L_list <- lapply(1:ncol(ctype_pairs), function(idx) {
    pair <- ctype_pairs[,idx]
    ctype1 <- gsub("Cell_type", "", pair[1])
    ctype2 <- gsub("Cell_type", "", pair[2])
    L <- matrix(
        0, nrow=length(coef_names), ncol=1, 
        dimnames=list(coef_names, paste0(ctype1, "_vs_", ctype2))
    )
    L[pair[1], 1] <-  1
    L[pair[2], 1] <- -1
    L
})
L <- do.call(cbind, L_list)

In [ ]:
# Batching contrasts

idx_list <- split(seq_len(ncol(L)), ceiling(seq_along(seq_len(ncol(L))) / 100))
res_list <- vector("list", length(idx_list))

for (i in seq_along(idx_list)) {
    print(paste("Starting batch", i))
    idx <- idx_list[[i]]
    Lsub <- L[,idx, drop=FALSE]
    fit  <- dream(vobj, form, sample_meta, L=Lsub, BPPARAM=param)
    batch_res <- lapply(colnames(Lsub), function(test) {
        tt <- topTable(fit, coef=test, number=Inf, p.value=0.05)
        tt$Test <- test
        tt
    })
    res_list[[i]] <- do.call(rbind, batch_res)
    rm(fit); gc()
}
pairwise_res <- do.call(rbind, res_list)
pairwise_res_list <- tapply(pairwise_res, pairwise_res$Test, list)

In [ ]:
names(pairwise_res_list)

Process pairwise DE results

In [ ]:
i <- grep("astrocyte_vs_VIP_GABAergic_cortical_interneuron", names(pairwise_res_list))
pairwise_res <- pairwise_res_list[[i]]

In [ ]:
pairwise_res %>% arrange(-logFC) %>% head()

In [ ]:
# Each test (e.g. CT1 and CT2 comparison) returns both positive and negative LFC DE genes
# To simplify downstream analyses, I want to split these out, e.g. make separate tables for CT1 vs. CT2 DE genes, and CT2 vs. CT1 DE genes
# This will require subsetting tables from a given comparison to genes with only positive (or negative) LFC values, depending on which cell type was the reference level

pairwise_res_list_split <- lapply(seq_along(pairwise_res_list), function(i) {
    pairwise_res <- pairwise_res_list[[i]]
    # Note: the first cell type in the test will always be the reference level...
    #   due to the fact each cell type was tested against other cell types that had names that came AFTER it (in alphabetical order),
    #   which is how factor levels get assigned
    ctype_res <- lapply(c(">", "<"), function(func_str) {
        func <- match.fun(func_str)
        pairwise_res[func(pairwise_res$logFC, 0),]
    })
    dir1_test <- names(pairwise_res_list)[i]
    ctypes <- unlist(strsplit(dir1_test, "_vs_"))
    test_names <- c(
        dir1_test, 
        paste0(ctypes[2], "_vs_", ctypes[1])
    )
    names(ctype_res) <- test_names
    ctype_res 
})

In [ ]:
pairwise_res_split <- unlist(pairwise_res_list_split, recursive=FALSE)

In [ ]:
saveRDS(pairwise_res_split, file="data/DE_genes/yao_2021_MOp_STAR_donor_cell_type_pseudobulk_pairwise_DE_genes_dream.RDS")